# 5 · Hooks firing in sequence — driven by the model's tool calls

A hook is a callback the agent invokes at a lifecycle boundary. In demo 4 we met
the machinery; here we watch it **fire for real**. The important shift: we do
**not** hand-fire hooks with fabricated payloads anymore. Instead we let the
**model call the tools itself**, and the agent **fires the hooks in order around
those calls** — `userpromptsubmit → responsereceived → pretooluse → posttooluse →
… → stop` — in the exact sequence it happens.

The loop's hook call sites live in `src/agent.rs`:

| Event | Where it fires in the loop |
|-------|----------------------------|
| `userpromptsubmit`  | once, when the prompt enters `Agent::run` |
| `responsereceived`  | once per round, when the model's raw response arrives, *before* any tool runs |
| `pretooluse`        | once per tool call, just *before* it executes |
| `posttooluse`       | once per tool call, *after* its result is produced |
| `stop`              | once, when the turn ends (`finish_reason = stop`) |
| `subagentstart` / `subagentstop` | bracketing a delegated `run_subagent` |

**Sequential is built in.** This loop runs tools one at a time by default
(`parallel_tools = false`), so `pretooluse` / `posttooluse` for tool *N+1* can
only fire after tool *N* has finished. The shared log below is the proof: each
entry records the event in the order the model's own decisions triggered it.

In [2]:
:dep agent_loop = { path = "/home/christian/Sandbox/agent-loop" }
:dep serde_json = "1"

use agent_loop::hooks::{Hooks, HookEvent, HookPayload};

// A Hooks registry with the shared log enabled. Every fired hook is appended.
let mut hooks = Hooks::new().with_log();

// One callback per event. Each prints a distinguishable line as it fires.
hooks.on_user_prompt_submit(|p| if let HookPayload::UserPromptSubmit { prompt } = p {
    let head: String = prompt.chars().take(50).collect();
    println!("▶ userpromptsubmit : user asked: {head}…");
});
// The key one for this demo: as soon as the model's response arrives — BEFORE
// any tool runs — print the full raw HTTP response, so you can see the
// tool_calls the model asked for right before they execute.
hooks.on_response(|p| if let HookPayload::ResponseReceived { response } = p {
    println!("▶ responsereceived : the model replied (raw HTTP response):");
    println!("{}", serde_json::to_string_pretty(response).unwrap());
    println!();
});
hooks.on_pre_tool_use(|p| if let HookPayload::PreToolUse { tool, args } = p {
    println!("▶ pretooluse      : about to call `{tool}` args={args}");
});
hooks.on_post_tool_use(|p| if let HookPayload::PostToolUse { tool, output, is_error } = p {
    let frag: String = output.chars().take(40).collect();
    println!("▶ posttooluse     : `{tool}` is_error={is_error} -> {frag}…");
});
hooks.on_stop(|p| if let HookPayload::Stop { reason } = p {
    println!("▶ stop            : turn finished with {reason}");
});
hooks.on_sub_agent_start(|p| if let HookPayload::SubAgentStart { name, .. } = p {
    println!("▶ subagentstart   : spawning sub-agent `{name}`");
});
hooks.on_sub_agent_stop(|p| if let HookPayload::SubAgentStop { name, .. } = p {
    println!("▶ subagentstop    : sub-agent `{name}` finished");
});

println!("attached callbacks: {}", hooks.count());

attached callbacks: 7


### 5.1 · Let the model call the tools — the agent fires hooks around them

The cell below runs a real `Agent::run` against the endpoint in your
environment. The system prompt asks the model to do a small multi-step job
(`calc`, then `bash`). Each step is the model's **own decision**; the loop only
*observes* it by firing the hooks.

Watch the **order**, *per tool the model asks for*:

1. `responsereceived` — the model's raw HTTP response arrives; you see the
   `tool_calls` it asked for (`name` + `arguments`);
2. `pretooluse` — the tool is *about to* execute;
3. `posttooluse` — the tool ran and returned.

So the response JSON is printed **before** the tool executes, for every tool.
When the model is done, the loop fires `stop`.

If no endpoint is reachable the cell prints a note instead of failing.

In [3]:
use agent_loop::tools::ToolRegistry;
use agent_loop::chat::{ChatClient, default_model};
use agent_loop::{Agent, AgentConfig};

let registry = ToolRegistry::with_builtins();
let client = ChatClient::from_env();
let mut agent = Agent::new(AgentConfig::new(
    default_model(),
    "You are a precise assistant. Follow the steps exactly:\n\
     1) use the calc tool for each arithmetic expression;\n\
     2) then use bash_run to echo a short confirmation;\n\
     3) finally answer with one terse line.",
    registry,
    hooks.clone(),
    client,
));
// Sequential on purpose: we want to SEE each pre/post pair fire one at a time.
agent.config.parallel_tools = false;

// Run the loop. Because of the `on_response` hook registered above, the model's
// raw response prints BEFORE its tool executes — for every tool it asks for.
let live: Option<agent_loop::AgentOutcome> = match agent.run(
    "Compute (2 + 3) * 4 with the calc tool, then run `echo hooks-fire-in-order` via bash, then tell me what you got.",
) {
    Ok(o) => { println!("final answer: {}", o.final_text); Some(o) }
    Err(e) => { println!("[no reachable endpoint] live run skipped: {e}"); None }
};

▶ userpromptsubmit : user asked: Compute (2 + 3) * 4 with the calc tool, then run `…
▶ responsereceived : the model replied (raw HTTP response):
{
  "choices": [
    {
      "finish_reason": "tool_calls",
      "index": 0,
      "message": {
        "content": "",
        "role": "assistant",
        "tool_calls": [
          {
            "function": {
              "arguments": "{\"expression\": \"(2 + 3) * 4\"}",
              "name": "calc"
            },
            "id": "chatcmpl-tool-a177c058849802e7",
            "type": "function"
          },
          {
            "function": {
              "arguments": "{\"command\": \"echo hooks-fire-in-order\"}",
              "name": "bash_run"
            },
            "id": "chatcmpl-tool-a06a09502232030a",
            "type": "function"
          }
        ]
      }
    }
  ],
  "id": "chatcmpl-R07EOwzTaOawNgAKdQMw6p8K",
  "model": "deepseek-v4-flash-0731",
  "usage": {
    "completion_tokens": 85,
    "prompt_tokens": 1083,
    "

### 5.2 · The model's reply → the hooks it fired

Line the two up in lock-step. The cell below replays the **raw response** the
model returned each round and, right beside it, the **hook entries** the loop
fired *because of* that reply — using the same global indices as the full log
that follows.

Each tool round consumes one `responsereceived` (the reply arrived, tool_calls
seen) plus a `pretooluse`/`posttooluse` pair (the tool executed). The final
round is a single `stop`.

In [4]:
if let Some(outcome) = &live {
    let all: Vec<String> = hooks.log_entries()
        .iter().map(|e| e.event.as_str().to_string()).collect();
    let mut idx = 0usize;

    if idx < all.len() && all[idx] == "userpromptsubmit" {
        println!("1. userpromptsubmit\n");
        idx += 1;
    }

    for (ti, t) in outcome.traces.iter().enumerate() {
        let n_calls = t.executed.len();
        let raw_calls = t.response
            .pointer("/choices/0/message/tool_calls")
            .and_then(|v| v.as_array());

        println!("── iteration {} · finish_reason={} ──", ti + 1, t.finish_reason);

        match raw_calls {
            Some(calls) if !calls.is_empty() => {
                println!("   model replied with {} tool_calls:", calls.len());
                for c in calls {
                    let name = c.pointer("/function/name").and_then(|v| v.as_str()).unwrap_or("?");
                    let args = c.pointer("/function/arguments").and_then(|v| v.as_str()).unwrap_or("{}");
                    println!("     • {name}   {args}");
                }
                println!("   → hook entries the loop fired for them (response, then pre/post per call):");
                // One `responsereceived` + a pre/post pair per call.
                let want = 1 + n_calls * 2;
                let take = want.min(all.len().saturating_sub(idx));
                for _ in 0..take {
                    println!("      {:2}. {}", idx + 1, all[idx]);
                    idx += 1;
                }
            }
            _ => {
                println!("   model replied directly, no tool_calls");
                if idx < all.len() {
                    println!("      {:2}. {}", idx + 1, all[idx]);
                    idx += 1;
                }
            }
        }
        println!();
    }
} else {
    println!("no live run data (endpoint was unreachable earlier).");
}

1. userpromptsubmit


── iteration 1 · finish_reason=tool_calls ──


   model replied with 1 tool_calls:


     • calc   {"expression": "(2 + 3) * 4"}


   → hook entries the loop fired for them (response, then pre/post per call):


...

       7. posttooluse


── iteration 3 · finish_reason=stop ──


   model replied directly, no tool_calls


       8. stop


()

### 5.3 · Sub-agent hooks — the bracket around a delegated task

The agent loop itself never spawns sub-agents, so `subagentstart` and
`subagentstop` come from `run_subagent`. It fires `subagentstart` just before
the child's one-shot call and `subagentstop` right after, so they bracket the
delegation exactly the way `pretooluse` / `posttooluse` bracket a tool. Fresh
registry and log so this section is its own clean sequence.

In [5]:
use agent_loop::agent::run_subagent;

let s_registry = ToolRegistry::with_builtins();
let mut s_hooks = Hooks::new().with_log();
s_hooks.on_sub_agent_start(|_| println!("▶ subagentstart: spawning"));
s_hooks.on_sub_agent_stop(|_| println!("▶ subagentstop : finished"));

let sub_name = "code_reviewer";
let _ = run_subagent(
    &AgentConfig::new(
        default_model(),
        "Answer briefly.",
        s_registry,
        s_hooks.clone(),
        ChatClient::from_env(),
    ),
    sub_name,
    "Name one risk in shipping a Rust binary without a test suite.",
);

println!();
for (i, entry) in s_hooks.log_entries().iter().enumerate() {
    println!("{:2}. {}", i + 1, entry.event.as_str());
}

▶ subagentstart: spawning


▶ subagentstop : finished


 1. subagentstart


 2. subagentstop


()

### 5.4 · Why this matters

Hooks only ever *observe*; they never steer the loop. The model called `calc`
and `bash` on its own initiative — the hook system didn't decide that, it just
recorded it, in order. That is the whole design: `pretooluse` gives you a veto
point *before* a tool runs, `posttooluse` gives you telemetry *after*, `stop`
closes the turn, and the sub-agent hooks bracket delegation. All additive, all
non-intrusive, and all fired on the same clock the model drives.